<a href="https://colab.research.google.com/github/dmainagithub/LLMs-Lessons/blob/main/huggingface_text_classification_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification Tutorial

Note: a GPU is needed in google colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU

## 2. Import necessary commands

In [50]:
# Install dependencies
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio # -U stands for upgrade
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")



Using transformers version: 5.16.1
Using datasets version: 5.0.1
Using torch version: 2.11.0+cu128


## 3. Getting a dataset

In [51]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [52]:
# What features are there
dataset.column_names

{'train': ['text', 'label']}

In [53]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [54]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [55]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f" Text: {text} | Label: {label}")


[52, 27, 218, 180, 213]
[INFO] Random samples from dataset:

 Text: Pizza with a unique, round-shaped crust hole in the middle | Label: food
 Text: A whole pizza pie with a thin and crispy crust | Label: food
 Text: Mouthwatering mushroom curry, featuring shiitake and button mushrooms in a rich coconut milk sauce with spices and herbs. | Label: food
 Text: A close-up of a girl feeding her rabbit in the garden | Label: not_food
 Text: Chandelier casting light in a dining room | Label: not_food


In [56]:
range(len(dataset["train"]))

range(0, 250)

In [57]:
dataset["train"].unique("label")

['food', 'not_food']

In [58]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])


Counter({'food': 125, 'not_food': 125})

In [59]:
# Turn our dataset into a dataframe
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
32,A bowl of sliced pears with a sprinkle of ging...,food
237,"Radishes in a bowl, sprinkled with salt and se...",food
28,"Mouthwatering paneer tikka masala, featuring j...",food
74,Set of paintbrushes stored in a jar,not_food
204,"Tangy tomato curry with chicken, featuring ten...",food
18,A pair of slices from a barbecue chicken pizza,food
42,Set of tongs stored in a drawer,not_food


In [60]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

1. Tokenization (machines prefer numbers rather than words)

2. Creating a train-test split (train split for training and test split for evaluation)

In [61]:
# Create a mapping programmatically
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: idx for idx, label in id2label.items()}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [62]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
  print(idx, label)
  id2label[idx] = label

0 not_food
1 food


In [63]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample ={"text": "This is a sentence about my favorite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favorite food: honey', 'label': 1}

In [64]:
# Map our dataset labels to numbers (the whole dataset)
# With dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}

In [69]:
# Shuffle data and look at 5 more random samples
dataset.shuffle()[:5]

{'text': ['Washing machine and dryer side by side in a laundry room',
  "Wooden cutting board with a chef's knife ready for use",
  'A close-up shot of a cheesy pizza slice being pulled away from the pie',
  'White porcelain sink with a shiny chrome faucet',
  "King-size bed with a white comforter inviting a good night's sleep"],
 'label': [0, 0, 1, 0, 0]}

### Train Test Split

* https://huggingface.co/docs/datasets/v4.8.4/process  

In [70]:
dataset = dataset.train_test_split(test_size=0.2, seed=42)
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 200
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 50
    })
})

In [71]:
random_idx_train = random.randint(0, len(dataset["train"]))
random_sample_train = dataset["train"][random_idx_train]
random_sample_train

{'text': 'Set of tea towels folded in a kitchen', 'label': 0}

In [73]:
random_idx_test = random.randint(0, len(dataset["test"]))
random_sample_test = dataset["train"][random_idx_test]
random_sample_test

{'text': 'A basket of fresh strawberries with a sprinkle of powdered sugar',
 'label': 1}

## Tokenization

https://platform.openai.com/tokenizer - open ai tokenizers.
https://github.com/huggingface/tokenizers - huggingface tokenizers.
To do this locally you need to have rust installed.
* RUST is a programming language: https://rust-lang.org/
* Huggingface auto classes: https://huggingface.co/docs/transformers/en/model_doc/auto

To find all the models: https://huggingface.co/models

We will use this specific one: https://huggingface.co/distilbert/distilbert-base-uncased

** Models are often paired with tokenizers.
* Tokenizers = turn text to numbers.
* Models = find patterns in those numbers.


In [76]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path="distilbert/distilbert-base-uncased",
                                          use_fast=True)
tokenizer

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

BertTokenizer(name_or_path='distilbert/distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [79]:
# Test our tokenizer
# Open ai token ids for "I love pizza" = [[40, 3047, 27941]]
tokenizer("I love pizza")

{'input_ids': [101, 1045, 2293, 10733, 102], 'token_type_ids': [0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1]}